In [ ]:
# Percorsi dei file.
import os
# Ricerca di file con caratteri jolly (curve di loss).
from glob import glob
# Espressioni regolari, per leggere seed e fold dai nomi dei file.
import re

# Grafici.
import matplotlib.pyplot as plt
# Calcolo numerico.
import numpy as np
# Tabelle.
import pandas as pd
# Mostra nel notebook le immagini salvate su disco.
from IPython.display import Image, display

# Grafico R² / Spearman / EF (passo 2).
from s2_metrics import r2_vs_ranking_plot, r2_comparison_plot
# Nomi delle isoforme e delle colonne target (passo 0).
from s0_config import ISOFORMS, TARGET_COLS

# Cartella dei risultati prodotti da main.ipynb.
OUTPUT_DIR = 'outputs'
DMPNN_OUT = os.path.join(OUTPUT_DIR, 'dmpnn')
COMPARISON_OUT = os.path.join(OUTPUT_DIR, 'comparison')


In [ ]:
# Stile comune dei grafici (lo stesso di main.ipynb).
plt.rcParams['figure.figsize'] = (13.33, 4)  # dimensione di default, in pollici
plt.rcParams['figure.dpi'] = 100  # risoluzione a schermo
plt.rcParams['savefig.dpi'] = 600  # risoluzione dei file salvati
plt.rcParams['font.size'] = 12  # dimensione del testo
plt.rcParams['axes.titlesize'] = 16  # dimensione dei titoli
plt.rcParams['axes.labelsize'] = 14  # dimensione delle etichette degli assi
plt.rcParams['legend.fontsize'] = 16  # dimensione del testo della legenda
plt.rcParams['lines.linewidth'] = 2  # spessore delle linee
plt.rcParams['lines.markersize'] = 4  # dimensione dei marcatori
plt.rcParams['axes.grid'] = False  # nessuna griglia di default
plt.rcParams['grid.alpha'] = 0.5  # trasparenza della griglia, quando attivata
plt.rcParams['grid.linestyle'] = '-.' # stile tratto-punto della griglia

In [ ]:
# Legge il dataset.
df = pd.read_excel('data.xlsx')

# Per ogni isoforma: numero di molecole misurate e frazione sul totale.
db = pd.DataFrame([(col.replace('_HUMAN', ''), df[col].count(), df[col].count() / len(df)) for col in TARGET_COLS],
                  columns=['HDAC', 'Count', 'Percentage'])
# Mostra la tabella.
print(db)

# Per ogni molecola, su quante isoforme è stata misurata.
measurements_per_molecule = df[TARGET_COLS].notna().sum(axis=1)
# Quante molecole hanno 1, 2, 3, … misure.
valid_distribution = measurements_per_molecule.value_counts().sort_index()
# Le classi da 1 a 4 misure restano separate…
grouped_distribution = valid_distribution[valid_distribution.index < 5].copy()
# …quelle con 5 o più vengono riunite in una sola fetta.
grouped_distribution['≥5'] = valid_distribution[valid_distribution.index >= 5].sum()
# Mostra i valori del grafico.
print('Grouped data for the chart:')  # intestazione
print(grouped_distribution)  # numero di molecole per fetta

# Grafico a torta.
plt.figure(figsize=(9, 8))
# Fette con percentuale, partendo dall'alto.
wedges, texts, autotexts = plt.pie(grouped_distribution,
                                   labels=grouped_distribution.index,  # etichetta = numero di misure
                                   autopct='%1.1f%%',  # percentuale con un decimale
                                   startangle=90,  # prima fetta in alto
                                   pctdistance=0.75,  # posizione delle percentuali
                                   textprops={'fontsize': 16})  # dimensione delle etichette
# Percentuali più grandi e in grassetto.
for autotext in autotexts:
    # Dimensione del testo.
    autotext.set_fontsize(18)
    # Grassetto.
    autotext.set_fontweight('bold')
# Sistema gli spazi.
plt.tight_layout()
# Mostra il grafico.
plt.show()

In [ ]:
# Sotto questa soglia di molecole di test l'R² di un'isoforma è poco affidabile.
UNDERPOWERED_N = 25
# Modelli da confrontare (nomi delle colonne nella tabella riassuntiva).
MODELS = ['LGBM', 'DMPNN', 'Stacking']

# Tabella riassuntiva scritta da main.ipynb (cella delle tabelle per isoforma), in ordine di isoforma.
df = pd.read_csv(os.path.join(COMPARISON_OUT, 'per_isoform_consolidated.csv')).set_index('isoform').reindex(ISOFORMS).reset_index()
# Molecole di test per isoforma.
n_test = df['n_test'].values

# Istogramma raggruppato dell'R² per isoforma (stessa funzione usata da main.ipynb).
r2_comparison_plot(df, os.path.join(COMPARISON_OUT, 'r2_per_isoform_comparison.png'), underpowered_n=UNDERPOWERED_N)
# Lo mostra nel notebook.
display(Image(filename=os.path.join(COMPARISON_OUT, 'r2_per_isoform_comparison.png')))

# R² dello stacking, con il nome di colonna atteso dalla funzione…
r2_df_stacking = df[['isoform', 'R2_Stacking']].rename(columns={'R2_Stacking': 'r2'})
# …e sue metriche di ranking.
ranking_stacking = pd.read_csv(os.path.join(COMPARISON_OUT, 'ranking_stacking.csv'))
# Grafico R² vs Spearman vs EF@10%.
r2_vs_ranking_plot(r2_df_stacking, ranking_stacking, output_path=os.path.join(COMPARISON_OUT, 'r2_vs_ranking_stacking.png'))
# Mostra anche questo.
display(Image(filename=os.path.join(COMPARISON_OUT, 'r2_vs_ranking_stacking.png')))

In [ ]:
# Curve della loss di validazione del D-MPNN, una per fold, smussate con una media mobile
# esponenziale (EMA). Scrive in outputs/loss/ un grafico per seed (tutti i fold) e uno per fold.

# Prefisso dei file di storia scritti dal training. I run fatti prima del riordino del
# codice usavano 'DMPNN_DMPNN': cambiare qui se si vogliono le curve di quei run.
# Finestra della media mobile esponenziale (in epoche).
W = 5
# Esponente della scala dell'asse y (vedi set_power_yscale).
K = 3.0
# Cartella delle immagini.
LOSS_DIR = os.path.join(DMPNN_OUT, 'loss_curves')
# Colori di default di matplotlib, uno per fold.
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# File di storia trovati: per ognuno legge seed e fold dal nome.
found = [re.search(r'seed(\d+)_cv_history_fold(\d+)\.csv$', f) for f in glob(os.path.join(DMPNN_OUT, 'seed*_cv_history_fold*.csv'))]
# Coppie (seed, fold) ordinate.
pairs = sorted((int(m.group(1)), int(m.group(2))) for m in found if m)
# Senza file non c'è niente da disegnare.
assert pairs, f'no {DMPNN_OUT}/seed*_cv_history_fold*.csv found: run main.ipynb first'
# Seed e fold presenti.
SEEDS, FOLDS = sorted({s for s, _ in pairs}), sorted({f for _, f in pairs})
# Riepilogo.
print(f'found seeds {SEEDS}, folds {FOLDS}')
# Crea la cartella delle immagini.
os.makedirs(LOSS_DIR, exist_ok=True)


# Asse y "a potenza": comprime la zona 0-0.8 ed espande quella alta, dove stanno le curve.
def set_power_yscale(ax):
    # Trasformazione y -> y^K e sua inversa.
    ax.set_yscale('function', functions=(lambda y: np.power(np.clip(y, 0, None), K),
                                         lambda y: np.power(np.clip(y, 0, None), 1.0 / K)))
    # Limiti e tacche dell'asse.
    ax.set_ylim(0.0, 1.15)  # limiti dell'asse
    ax.set_yticks([0.0, 0.4, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1])  # tacche più fitte nella zona alta


# Legge la storia di un fold: epoche, MSE di validazione grezzo e sua EMA.
def load_fold(seed, fold):
    # Tabella delle epoche.
    h = pd.read_csv(os.path.join(DMPNN_OUT, f'seed{seed}_cv_history_fold{fold}.csv'))
    # MSE di validazione per epoca.
    val = h['val_mse'].values
    # Epoche, valori grezzi e media mobile esponenziale.
    return h['epoch'].values, val, pd.Series(val).ewm(span=W, adjust=False).mean().values


# Disegna in ax il fold i-esimo: curva grezza chiara e curva EMA spessa.
def plot_fold(ax, seed, fold, i, alpha_raw):
    # Dati del fold.
    ep, val, ema = load_fold(seed, fold)
    # Colore del fold.
    c = colors[i % len(colors)]
    # Curva grezza, sottile e trasparente.
    ax.plot(ep, val, color=c, alpha=alpha_raw, linewidth=1.0)
    # Curva smussata, con il minimo in legenda.
    ax.plot(ep, ema, color=c, linewidth=2.4, label=f'fold {fold} (val_min={ema.min():.3f})')


# Rifinisce e salva una figura con sfondo trasparente.
def save(fig, ax, path):
    # Etichette degli assi.
    ax.set_xlabel('epoch'); ax.set_ylabel('Validation MSE')
    # Scala a potenza, griglia e legenda.
    set_power_yscale(ax); ax.grid(True, alpha=0.3); ax.legend(loc='upper right', framealpha=0.9)
    # Spazi, sfondo trasparente e salvataggio.
    fig.tight_layout(); ax.patch.set_alpha(0.0); fig.savefig(path, dpi=150, transparent=True)
    # Chiude la figura (non la mostra) e conferma.
    plt.close(fig); print('saved', path)


# Per ogni seed…
for seed in SEEDS:
    # …un grafico con tutti i fold…
    fig, ax = plt.subplots(figsize=(13, 6.5))
    # …una coppia di curve per fold…
    for i, fold in enumerate(FOLDS):
        # …disegnata nello stesso riquadro.
        plot_fold(ax, seed, fold, i, alpha_raw=0.22)
    # Titolo del grafico del seed.
    ax.set_title(f'Validation loss D-MPNN — seed {seed}, EMA (w={W}) per fold')
    # Salvataggio.
    save(fig, ax, os.path.join(LOSS_DIR, f'seed{seed}_all_folds.png'))
    # …e un grafico per ogni fold.
    for i, fold in enumerate(FOLDS):
        # Figura nuova.
        fig, ax = plt.subplots(figsize=(13, 6.5))
        # Curve del solo fold.
        plot_fold(ax, seed, fold, i, alpha_raw=0.25)
        # Titolo con seed e fold.
        ax.set_title(f'Validation loss D-MPNN — seed {seed}, fold {fold}, EMA (w={W})')
        # Salvataggio.
        save(fig, ax, os.path.join(LOSS_DIR, f'seed{seed}_fold{fold}.png'))